# Gradient Boosting Experiment

This notebook tunes Gradient Boosting using the same Phase 3 feature contract and Phase 4 search helper, then evaluates ranking, classification, calibration proxy, and feature importance behavior.

In [1]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
ROOT = Path.cwd()
while not (ROOT / 'ml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from ml.preprocessing.build_pipeline import PIPELINE_FILENAME
from ml.preprocessing.clean_data import TARGET_COLUMN
from ml.preprocessing.split_data import RANDOM_SEED
from ml.training.train_models import build_model_searches, _balanced_sample_weights, _classification_metrics
ARTIFACTS = ROOT / 'ml' / 'artifacts'
PROCESSED = ROOT / 'ml' / 'data' / 'processed'
sns.set_theme(style='whitegrid')

## Load and transform the cleaned splits
No preprocessing statistics are refit in this experiment.

In [ ]:
preprocessor = joblib.load(ARTIFACTS / PIPELINE_FILENAME)
train = pd.read_csv(PROCESSED / 'train.csv')
validation = pd.read_csv(PROCESSED / 'validation.csv')
test = pd.read_csv(PROCESSED / 'test.csv')
x_train = preprocessor.transform(train.drop(columns=[TARGET_COLUMN]))
x_validation = preprocessor.transform(validation.drop(columns=[TARGET_COLUMN]))
x_test = preprocessor.transform(test.drop(columns=[TARGET_COLUMN]))
y_train = train[TARGET_COLUMN].astype(int)
y_validation = validation[TARGET_COLUMN].astype(int)
y_test = test[TARGET_COLUMN].astype(int)
print({'train': x_train.shape, 'validation': x_validation.shape, 'test': x_test.shape})

## Cross-validation and hyperparameter tuning
The reusable search uses stratified three-fold CV and ROC-AUC. Balanced sample weights emphasize the minority class during fitting.

In [ ]:
search = build_model_searches(RANDOM_SEED)['gradient_boosting']
search.fit(x_train, y_train, sample_weight=_balanced_sample_weights(y_train))
model = search.best_estimator_
cv_summary = pd.DataFrame({'best_cv_roc_auc': [search.cv_results_['mean_test_score'][search.best_index_]], 'cv_roc_auc_std': [search.cv_results_['std_test_score'][search.best_index_]], 'best_parameters': [search.best_params_]})
display(cv_summary)

## Evaluation metrics

In [ ]:
def metric_table(features, target, split_name):
    probabilities = model.predict_proba(features)[:, 1]
    return pd.Series(_classification_metrics(target, probabilities), name=split_name)
metrics = pd.concat([metric_table(x_validation, y_validation, 'validation'), metric_table(x_test, y_test, 'test')], axis=1)
display(metrics.round(4))

## Visual diagnostics and feature importance
Tree importance is aggregated to original input fields so one-hot categories do not dominate the display merely because they expand into multiple columns.

In [ ]:
validation_probabilities = model.predict_proba(x_validation)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(y_validation, validation_probabilities >= 0.5, ax=axes[0], cmap='Blues')
axes[0].set_title('Validation confusion matrix')
RocCurveDisplay.from_predictions(y_validation, validation_probabilities, ax=axes[1])
axes[1].set_title('Validation ROC curve')
plt.tight_layout()
plt.show()
feature_metadata = json.loads((ARTIFACTS / 'feature_metadata.json').read_text())
mapping = feature_metadata['transformation_mapping']
importance = pd.Series(model.feature_importances_, index=feature_metadata['transformed_feature_names'])
original_importance = importance.groupby(importance.index.map(mapping)).sum().sort_values(ascending=False).head(20).sort_values()
original_importance.plot.barh(figsize=(9, 7), color='#c45b4d')
plt.title('Gradient Boosting importance by original feature')
plt.xlabel('Aggregated tree importance')
plt.show()

## Observations
- Gradient Boosting is sensitive to depth, learning rate, and estimator count, so the tuned parameters should be recorded with the metrics.
- Feature importance is a model-internal association and should not be described as causal evidence.
- Thresholds and the primary model remain Phase 5 decisions; this notebook does not manufacture them.